# Agri Vision: native YOLOv8n, rounds 1–30

Use your **separate YOLO account**. Upload `agrivision_30_round_code.zip` and the existing `non_chili_v1_colab_dataset.zip` to `MyDrive/Agri Vision/`. The same verified 14-part dataset folder is also supported. The 24-class data remains unchanged.

Choose a **T4 GPU**. **Run all** first performs a small training-only learning check, then starts the fresh 30-round main run if the check passes. Rerunning resumes the last completed global round automatically; an interrupted partial round repeats. Pilot weights are never used for the main run.

Native Ultralytics 8.4.48 training restores the historical method: three clients, one local epoch, batch 8, 640 images, native optimizer/warmup/augmentation and adapting BatchNorm. Seed 42 and the new 24-class dataset are declared differences from the old experiment. Metrics and initial/best/latest weights are saved; bulky client weights and every-round raw prediction exports are not retained.


In [ ]:
from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/Agri Vision")
WORKSPACE = Path("/content/agrivision_30_round")
EXPECTED_DATASET_SHA256 = "7a2920e4a4e0405ee157e9b40ac6306b54d6933b20a306b44a4b7c6871f20868"
EXPECTED_CODE_SHA256 = "a1fe67c876bb3a2e5da123e9f5820652e91189f8fe3b21cead34381c5422f579"
TARGET_ROUNDS = 30
SEED = 42
WORKERS = 2


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import hashlib, json, shutil, zipfile
def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream,"sha256").hexdigest()
WORKSPACE.mkdir(parents=True, exist_ok=True)
def upload_path(filename):
    path = DRIVE_ROOT / filename
    if path.is_file() or filename != "non_chili_v1_colab_dataset.zip":
        return path
    parts_dir = DRIVE_ROOT / "non_chili_v1_upload_parts"
    manifest_path = parts_dir / "parts.json"
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text())
        assert manifest["joined_sha256"] == EXPECTED_DATASET_SHA256, "Wrong dataset parts manifest"
        specs = manifest["parts"]
        assert specs and [item["name"] for item in specs] == [filename + f".part{index:02d}" for index in range(1, len(specs) + 1)], "Missing or incorrectly ordered parts manifest"
        parts = [parts_dir / item["name"] for item in specs]
        assert all(part.is_file() and part.stat().st_size == item["bytes"] for part, item in zip(parts, specs)), "Upload all listed dataset parts with their complete sizes first"
    else:
        parts = [DRIVE_ROOT / (filename + suffix) for suffix in (".part01", ".part02")]
        assert all(part.is_file() for part in parts), "Upload the dataset ZIP or the dataset parts folder first"
    joined = WORKSPACE.parent / filename
    if joined.is_file() and sha(joined) == EXPECTED_DATASET_SHA256:
        return joined
    temporary = joined.with_suffix(joined.suffix + ".partial")
    with temporary.open("wb") as output:
        for part in parts:
            with part.open("rb") as source:
                shutil.copyfileobj(source, output, length=16 * 1024 * 1024)
    assert sha(temporary) == EXPECTED_DATASET_SHA256, "Dataset parts are damaged or do not belong together"
    temporary.replace(joined)
    print(f"Joined {len(parts)} dataset parts on Colab's local disk.")
    return joined
for filename, expected in [("non_chili_v1_colab_dataset.zip", EXPECTED_DATASET_SHA256),
                           ("agrivision_30_round_code.zip", EXPECTED_CODE_SHA256)]:
    path = upload_path(filename)
    assert path.is_file(), f"Upload {filename} first"
    assert sha(path) == expected, f"Wrong or damaged upload: {filename}"
    with zipfile.ZipFile(path) as archive:
        for name in archive.namelist():
            assert (WORKSPACE / name).resolve().is_relative_to(WORKSPACE.resolve()), "Unsafe archive path"
        archive.extractall(WORKSPACE)
print("Verified and extracted both uploads.")


In [ ]:
import os, subprocess, sys
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))
# Protect Colab's CUDA-enabled torch/torchvision pair from dependency replacement.
import torch, torchvision
torch_before, vision_before = torch.__version__, torchvision.__version__
assert torch.cuda.is_available(), "Select Runtime -> Change runtime type -> GPU"
gpu = torch.cuda.get_device_name(0)
print("GPU:", gpu, "torch:", torch_before, "torchvision:", vision_before)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-r", "requirements-comparison.txt"], check=True)
# Optional model dependencies, pinned where tested. --no-deps above preserves core CUDA packages.
subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub", "safetensors", "packaging", "tqdm", "omegaconf==2.4.0",
                "polars>=0.20", "ultralytics-thop>=2.0.18", "matplotlib", "opencv-python", "requests", "scipy", "psutil"], check=True)
assert torch.__version__ == torch_before and torchvision.__version__ == vision_before
for pattern in ["test_comparison.py", "test_30_round_workflow.py"]:
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", pattern, "-v"], check=True)


## Learning check and recoverable main run
The pilot uses 48 training images for 12 small epochs with augmentation disabled, and evaluates only those training images. Its scores are diagnostics. Main training starts from fresh pretrained initialization with the historical augmentation settings. A failed pilot stops this notebook.


In [ ]:
import json
from datetime import datetime, timezone
from comparison.data import load_protocol
from scripts.yolo_native_train import pipeline_hash, hardware
load_protocol(WORKSPACE)
assert "T4" in gpu, "Select a T4 for the agreed historical GPU environment"
OUTPUT_ROOT = DRIVE_ROOT / "comparison_results"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN = OUTPUT_ROOT / "yolo_v8n_native_non_chili_seed42_30round"
PILOT_RECORD = OUTPUT_ROOT / "yolo_native_learning_pilot_seed42.json"
# Resume does not repeat the pilot or initialize a new run.
if not RUN.exists():
    passed = False
    if PILOT_RECORD.exists():
        record = json.loads(PILOT_RECORD.read_text())
        passed = (record.get("phase") == "passed" and record.get("pipeline_hash") == pipeline_hash()
                  and record.get("hardware") == hardware(torch.device("cuda:0"))
                  and record.get("dataset_sha256") == load_protocol(WORKSPACE)["dataset_checksums_sha256"]
                  and record.get("seed") == SEED)
    if not passed:
        stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
        pilot = OUTPUT_ROOT / f"yolo_native_learning_pilot_{stamp}"
        subprocess.run([sys.executable, "-m", "scripts.yolo_native_train", "--root", str(WORKSPACE),
                        "--pilot", "--seed", str(SEED), "--output", str(pilot)], check=True)
        shutil.copy2(pilot / "pilot.json", PILOT_RECORD)
subprocess.run([sys.executable, "-m", "scripts.yolo_native_train", "--root", str(WORKSPACE),
                "--rounds", str(TARGET_ROUNDS), "--seed", str(SEED), "--pilot-record", str(PILOT_RECORD),
                "--output", str(RUN)], check=True)
print("YOLO run:", RUN)
print(json.loads((RUN / "latest.json").read_text()))


## Progress and results
While training, the output prints the round, client, native losses and validation. In Drive, `status.json` gives the phase; `latest.json` is the committed global round. The main CSV is `results/metrics/global_validation_metrics.csv`. Each round's JSON includes pooled comparison metrics and separately labelled native YOLO client metrics. Best selection uses pooled validation AP50–95, not the final round automatically. Held-out test evaluation is a separate action after configuration selection.

If the runtime stops, reopen this same notebook on the same account with the same uploaded bundle, select T4, and Run all. Do not delete the run folder or change the budget/packages. The weights, configuration, hardware and hashes are checked before resume.
